In [16]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import os

Load Data

In [13]:
input_path = "D:/Projects with Git/AI-For-Health-IITGN-SRIP/Dataset"

patient_paths = [f.path for f in os.scandir(input_path) if f.is_file() and f.name.endswith('_dataset.npz')]

datasets = {}

for path in patient_paths:
    patient_id = os.path.basename(path).replace('_dataset.npz', '')

    with np.load(path, allow_pickle=True) as data:
        datasets[patient_id] = {
            "X": data["X"],
            "y_sleep": data["y_sleep"],
            "y_breath": data["y_breath"]
        }

Prepare categorical to numerical mapping

In [26]:
sleep_set = set()
breath_set = set()

for patient in datasets:
    sleep_set.update(np.unique(datasets[patient]["y_sleep"]))
    breath_set.update(np.unique(datasets[patient]["y_breath"]))

sleep_set = sorted(sleep_set)
breath_set = sorted(breath_set)

print("\nUnique Categories")
print(f"Sleep States:{sleep_set}")
print(f"Breathing Irregularities:{breath_set}")

sleep_map = {
    " A": 0,
    " N1": 1,
    " N2": 2,
    " N3": 3,
    " Movement": 4,
    " REM": 5,
    " Wake": 6
}

breath_map = {
    "Body event": 0,
    "Hypopnea": 1,
    "Mixed Apnea": 2,
    "Normal": 3,
    "Obstructive Apnea": 4
}

breath_map_inverse = {value:key for key, value in breath_map.items()}

print("\nError/Miss Check")
print(f"Sleep States: {set(sleep_set) - set(sleep_map.keys())}")
print(f"Breathing Irregularities: {set(breath_set) - set(breath_map.keys())}")


Unique Categories
Sleep States:[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Breathing Irregularities:[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Error/Miss Check
Sleep States: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)}
Breathing Irregularities: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}


Normalize the columns in X, apply mapping to y_breath and y_sleep.

\# Not used

\# Normalization should be applied during cross validation, only on the training set

```python
for key in datasets.keys():
    datasets[key]["X"][:, :, 0] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 0], np.min(datasets[key]["X"][:, :, 0])), 
        np.max(datasets[key]["X"][:, :, 0]) - np.min(datasets[key]["X"][:, :, 0])
        )
    
    datasets[key]["X"][:, :, 1] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 1], np.min(datasets[key]["X"][:, :, 1])), 
        np.max(datasets[key]["X"][:, :, 1]) - np.min(datasets[key]["X"][:, :, 1])
        )
    
    datasets[key]["X"][:, :, 2] = np.divide(
        np.subtract(datasets[key]["X"][:, :, 2], np.min(datasets[key]["X"][:, :, 2])), 
        np.max(datasets[key]["X"][:, :, 2]) - np.min(datasets[key]["X"][:, :, 2])
        )
    
    datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
    datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])
```



In [15]:
# Applying mappings to y columns
try:
    for key in datasets.keys():
        datasets[key]["y_breath"] = np.vectorize(lambda x: breath_map[x])(datasets[key]["y_breath"])
        datasets[key]["y_sleep"] = np.vectorize(lambda x: sleep_map[x])(datasets[key]["y_sleep"])
except KeyError as e:
    print(f"Mapping Error; Already Mapped: {e}")

Initialize DL framework

In [6]:
# constants
TIME_STEPS = 960
FEATURES = 3
NUM_SLEEP_STATES = len(sleep_map)
NUM_CLASSES = len(breath_map)

def build_model():

    # ----- SIGNAL INPUT BRANCH -----
    signal_input = layers.Input(shape=(TIME_STEPS, FEATURES), name="signal_input")

    x = layers.Conv1D(32, 3, activation='relu', padding='same')(signal_input)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)

    signal_features = x


    # ----- SLEEP INPUT BRANCH -----
    sleep_input = layers.Input(shape=(1,), name="sleep_input")

    s = layers.Embedding(NUM_SLEEP_STATES, 4)(sleep_input)
    s = layers.Flatten()(s)

    sleep_features = s


    # ----- MERGE BRANCHES -----
    merged = layers.Concatenate()([signal_features, sleep_features])


    # ----- CLASSIFIER -----
    x = layers.Dense(64, activation='relu')(merged)
    x = layers.Dropout(0.3)(x)

    output = layers.Dense(NUM_CLASSES, activation='softmax', name="breath_output")(x)


    # ----- MODEL OBJECT -----
    model = Model(
        inputs=[signal_input, sleep_input],
        outputs=output
    )


    # ----- COMPILE -----
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ signal_input        │ (None, 960, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 960, 32)   │        320 │ signal_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 480, 32)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 480, 64)   │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 240, 64)   │          0 │ conv1d_1[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sleep_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 240, 128)  │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 4)      │         28 │ sleep_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4)         │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 132)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ breath_output       │ (None, 5)         │        325 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,097 (156.63 KB)

 Trainable params: 40,097 (156.63 KB)

 Non-trainable params: 0 (0.00 B)

Prepare for Model Training

In [20]:
# Leave One Out Cross Validation
# from sklearn.model_selection import LeaveOneGroupOut
# logo = LeaveOneGroupOut()

all_y_breath_true = []
all_y_breath_pred = []

for patient_id in datasets.keys():
    X_test = datasets[patient_id]["X"]
    y_sleep_test = datasets[patient_id]["y_sleep"]
    y_breath_test = datasets[patient_id]["y_breath"]

    train_ids = [pid for pid in datasets.keys() if pid != patient_id]
    X_train = np.concatenate([datasets[pid]["X"] for pid in train_ids], axis=0)
    y_sleep_train = np.concatenate([datasets[pid]["y_sleep"] for pid in train_ids], axis=0)
    y_breath_train = np.concatenate([datasets[pid]["y_breath"] for pid in train_ids], axis=0)

    # Normalization
    scaler_0 = StandardScaler()
    scaler_1 = StandardScaler()
    scaler_2 = StandardScaler()

    X_train[0] = scaler_0.fit_transform(X_train[0])
    X_train[1] = scaler_1.fit_transform(X_train[1])
    X_train[2] = scaler_2.fit_transform(X_train[2])

    X_test[0] = scaler_0.transform(X_test[0])
    X_test[1] = scaler_1.transform(X_test[1])
    X_test[2] = scaler_2.transform(X_test[2])

    print(f"\nTraining on {len(train_ids)} patients, Testing on {patient_id}")

    model.fit(
        [X_train, y_sleep_train],
        y_breath_train
    )

    y_pred = model.predict([X_test, y_sleep_test])
    y_pred_classes = np.argmax(y_pred, axis=1)

    report = classification_report(y_breath_test, y_pred_classes)
    
    all_y_breath_true.extend(y_breath_test)
    all_y_breath_pred.extend(y_pred_classes)



Training on 4 patients, Testing on AP01
219/219 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.9046 - loss: 0.3183
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step

Training on 4 patients, Testing on AP02


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


220/220 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.9135 - loss: 0.2845
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step

Training on 4 patients, Testing on AP03


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


222/222 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.8953 - loss: 0.3322
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step

Training on 4 patients, Testing on AP04


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - accuracy: 0.9138 - loss: 0.2840
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Training on 4 patients, Testing on AP05
226/226 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.9399 - loss: 0.2108
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step


c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Displaying Results

In [ ]:
# Calculate weighted average classification report

all_y_breath_true = np.array(all_y_breath_true)
all_y_breath_pred = np.array(all_y_breath_pred)

final_report_dict = classification_report(all_y_breath_true, 
                                          all_y_breath_pred, 
                                          digits=4, 
                                          output_dict=True)

mapped_report = {}
for label, metrics in final_report_dict.items():
    if label.isdigit():
        mapped_report[breath_map_inverse[int(label)]] = metrics
    else:
        mapped_report[label] = metrics  # macro avg, weighted avg, accuracy

# Put in DataFrame with desired row order
row_order = list(breath_map.keys()) + ["macro avg", "weighted avg", "accuracy"]
df_report = pd.DataFrame(mapped_report).T.reindex(row_order)
# ensure support is int for print formatting
if "support" in df_report.columns:
    df_report["support"] = df_report["support"].astype(int)

# Build printed rows manually for perfect control
print("               precision    recall  f1-score   support")
for idx in row_order:
    if idx == "macro avg":
        # blank line before macro avg
        print()
    row = df_report.loc[idx]
    if idx == "accuracy":
        # accuracy row prints with placeholder fields for recall/f1
        print(f"{idx:<16s} {row['precision']:.4f}      -     -   {int(row['support'])}")
    else:
        print(f"{idx:<16s} {row['precision']:.4f}    {row['recall']:.4f}    {row['f1-score']:.4f}   {int(row['support'])}")



c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


               precision    recall  f1-score   support


ValueError: Unknown format code 'd' for object of type 'float'